In [81]:
import torch
import copy

from huggingface_hub import login
from datasets import load_dataset
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig, get_peft_model_state_dict, set_peft_model_state_dict

from torch.utils.data import Dataset, DataLoader

from typing import TypedDict

In [83]:
login()

In [84]:
smol_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM3-3B")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

Loading weights: 100%|██████████| 326/326 [00:00<00:00, 13527.07it/s]


In [85]:
# Init server model
server_model = copy.copy(smol_model)
base_peft_model = get_peft_model(server_model, peft_config=lora_config)

# This needs to be sent to each client to start federation.
global_adapters = get_peft_model_state_dict(server_peft_model)


In [109]:
print(global_adapters["base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight"])

tensor([[-0.0177,  0.0024,  0.0176,  ...,  0.0131,  0.0183, -0.0197],
        [ 0.0132, -0.0099, -0.0046,  ...,  0.0041, -0.0037,  0.0143],
        [-0.0208,  0.0206, -0.0097,  ...,  0.0027, -0.0173, -0.0168],
        ...,
        [-0.0203,  0.0107,  0.0200,  ..., -0.0093, -0.0170, -0.0111],
        [-0.0004, -0.0027,  0.0154,  ..., -0.0165,  0.0016, -0.0171],
        [ 0.0047, -0.0138, -0.0187,  ..., -0.0055, -0.0220, -0.0060]])


In [110]:
class FL_CLient(TypedDict):
    client_name: str
    lora_sd: dict[str, torch.Tensor]
    data_loader: torch.utils.data.DataLoader
    num_samples: int

In [87]:
def fed_avg(keys, client_adapters: list[FL_CLient]):

    total_samples = 0
    for c in client_adapters:
        total_samples += c.num_samples

    for k in keys:
        shape = c.lora_sd[k].shape #tensor shape for specific key
        weighted_key_total = torch.zeros((shape))
        for c in client_adapters:
            weighted_key_total += (c.lora_sd[k] * (c.num_samples / total_samples)) # weight is number of training samples / total number of samples
        global_adapters[k] = weighted_key_total / len(client_adapters)

In [89]:
# Datasets
class Dreaddit(Dataset):
    def __init__(self, texts, text_tokenizer):
        self.texts = texts
        self.tokenizer = text_tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = self.texts[idx]

        prompt = item["prompt"]
        response = item["response"]
        full_prompt = f"{prompt} {response}"

        tokenizer_data = self.tokenizer(full_prompt).data

        # Raw token values
        input_ids = tokenizer_data["input_ids"]

        #Determines what a real token is
        attn_mask = tokenizer_data["attention_mask"]

        # Determines what gets scored
        labels = input_ids.clone()
        labels[:, :len(prompt)] = -100

        return input_ids, attn_mask, labels

class IRFDataset(Dataset):
    def __init__(self, toeknizer, texts):
        super().__init__()
        self.tokenizer = tokenizer
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = self.texts[idx]

        prompt = item["prompt"]
        response = item["response"]
        full_prompt = f"{prompt} {response}"


        tokenized = self.tokenizer(full_prompt).data

        input_ids = tokenized["input_ids"]
        attn_mask = tokenized["attention_mask"]

        labels = input_ids.clone()
        labels[: , :len(prompt)] = -100

        return input_ids, attn_mask, labels

In [90]:
# Map Functions
def build_dreaddit_features(sample):
    stress = sample["label"]
    response = STRESS_SUPPORT_STATEMENT if stress == 1 else GENERAL_SUPPORT_STATEMENT
    prompt = f"instruction:{BASE_INSTRUCTION} post:{sample["text"]} response: "

    return {"prompt": prompt, "response": response}

def build_irf_features(sample):
    belong = sample["belong"]
    burden = sample["burden"]
    post = sample["text"]

    if belong == 0 and burden == 0:
        target = irf_tuning_statements["0"]

    if belong == 1 and burden == 0:
        target = irf_tuning_statements["1"]

    if belong == 0 and burden == 1:
        target = irf_tuning_statements["2"]

    if belong == 1 and burden == 1:
        target = irf_tuning_statements["3"]

    return { "prompt":f"{irf_instruction} {post}", "response": target}


In [97]:
# Dreaddit
STRESS_SUPPORT_STATEMENT = "It sounds like you are carrying a lot of stress and feeling completely exhausted. Please remember to be gentle with yourself today, and consider stepping away for a short break to rest."

GENERAL_SUPPORT_STATEMENT = "Thank you for sharing your thoughts today. I hope things continue to go smoothly for you and that you have a wonderful, peaceful day ahead!"

BASE_INSTRUCTION = "Analyze this Reddit post for signs of psychological stress. Provide a brief, supportive response in 1–2 sentences. If stress is present, acknowledge feelings and suggest one helpful action. If no stress, provide encouragement. Be empathetic but avoid medical advice. Post: "

ds = load_dataset("hutchii/dreaddit")

train = ds["train"]
prompts = train.map(build_dreaddit_features).select_columns(column_names=["prompt", "response"])
prompt_data = prompts.data.to_pylist()

dreaddit = Dreaddit(texts=prompt_data, text_tokenizer=tokenizer)
dread_dl = DataLoader(dataset=dreaddit, batch_size=32, collate_fn=collate_fn)

dread_client: FL_CLient = {
    "client_name":"dreaddit",
    "data_loader":dread_dl,
    "num_samples": len(dread_dl)
}

In [96]:
#IRF
irf_dataset = load_dataset("hutchii/InterpersonalRiskFactors")
# belong:0 burden:0 - 0
# belong:1 burden:0 - 1
# belong:0 burden:1 - 2
# belong:1 burden:1 - 3

irf_tuning_statements = {
    "0":"Thank you for opening up and sharing your thoughts today. I hope things continue to go smoothly for you and that you have a peaceful, restful week ahead!",
    "1":"I hear how painful and exhausting it feels to be disconnected right now, but please know that your feelings are valid and you do not have to face this isolation alone. Consider reaching out to a trusted friend or community space when you feel ready.",
    "2":"It sounds like you are carrying a heavy emotional weight and feeling like a hardship to those around you, but your presence has genuine value. Please be gentle with yourself today and remember that needing support does not make you a burden.",
    "3":"I hear how completely overwhelming things feel right now, especially when experiencing both deep isolation and the feeling of being a burden to others. Please know that you matter and you do not have to carry this alone—consider connecting with a supportive person or crisis resource today."
}

irf_instruction = "Analyze this post for signs of interpersonal risk factors, specifically Thwarted Belongingness (feeling isolated, lonely, or disconnected) and Perceived Burdensomeness (feeling like a drain or weight on others). Provide a brief, supportive response in 1–2 sentences addressing these feelings. Be empathetic and encouraging while avoiding medical advice. Post: "

iff_train = irf_dataset["train"]
train_data = iff_train.map(build_irf_features).select_columns(column_names=["prompt", "response"]).data.to_pylist()

irf_dataset = IRFDataset(texts=train_data, toeknizer=tokenizer)
irf_dl = DataLoader(dataset=irf_dataset, batch_size=32, collate_fn=collate_fn)

irf_client:FL_CLient = {
    "client_name":"irf",
    "data_loader":irf_dl,
    "num_samples": len(irf_dl)
}

In [94]:
clients = [dread_client, irf_client]

In [ ]:
# ------- 1 ROUND----------
# 1. Send LoRA adapters from server model
# 2. Each client attaches adapters to baseline model
# 3. Clients train locally for two epocs
# 4. Clients add noise
# 5. Transmit to client
# 6. FedAvg performed on the server

In [ ]:
#Federated Training Loop
MAX_CLIENT_EPOCH = 2
MAX_FED_ROUNDS = 5
optimizer = torch.optim.Adam(peft_model.parameters(), lr=.01)
loss_fn = CrossEntropyLoss()

for r in range(MAX_FED_ROUNDS):
    client_adapters = []
    for c in clients:
        model = copy.copy(base_peft_model)
        set_peft_model_state_dict(model, global_adapters)

        for e in range(MAX_CLIENT_EPOCH):
            for x,y in c.data_loader:
                model.train()
                resp_tokens = model(x)
                loss = loss_fn(resp_tokens, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        # send lora adapter
        client_adapters.append(model.get_peft_model_state_dict)
    global_adapters = fed_avg(client_adapters)


